# Snap pickups to road edges and aggregate demand

Loads TLC 2010 Manhattan pickups, snaps each point to the nearest OSMnx road edge in chunks, joins pickup counts onto the Manhattan drive network, and saves `data/processed/edges_with_demand.gpkg`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import osmnx as ox

TARGET_CRS = "EPSG:2263"
CHUNK_SIZE = 100_000

# --- paths (works from repo root or notebooks/) ---
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PICKUPS_PATH = PROJECT_ROOT / "data/processed/tlc_2010_manhattan_pickups.parquet"
OUT_PATH = PROJECT_ROOT / "data/processed/edges_with_demand.gpkg"

def _first_existing(*paths: Path) -> Path:
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these paths exist: {', '.join(map(str, paths))}")

ROADS_PATH = _first_existing(
    PROJECT_ROOT / "data/raw/manhattan_roads.gpkg",
    PROJECT_ROOT / "notebooks/data/raw/manhattan_roads.gpkg",
)
NODES_PATH = _first_existing(
    PROJECT_ROOT / "data/raw/manhattan_nodes.gpkg",
    PROJECT_ROOT / "notebooks/data/raw/manhattan_nodes.gpkg",
)


def _load_graph_from_gpkg(roads_path: Path, nodes_path: Path, target_crs: str):
    """Rebuild OSMnx graph from saved node/edge GeoPackages in target_crs."""
    edges = gpd.read_file(roads_path)
    nodes = gpd.read_file(nodes_path)

    if edges.crs is None or nodes.crs is None:
        raise ValueError("Road network GeoPackages must define a CRS.")
    if str(edges.crs) != target_crs:
        edges = edges.to_crs(target_crs)
    if str(nodes.crs) != target_crs:
        nodes = nodes.to_crs(target_crs)

    # graph_from_gdfs expects x/y to match geometry after reprojection
    nodes["x"] = nodes.geometry.x
    nodes["y"] = nodes.geometry.y

    nodes = nodes.set_index("osmid", drop=False)
    edges_idx = edges.set_index(["u", "v", "key"])

    G = ox.graph_from_gdfs(nodes, edges_idx)
    return G, edges


# 1. Load pickups if not already in memory
if "gdf_mht_pu" in globals():
    gdf_pickups = globals()["gdf_mht_pu"]
    print(f"Using in-memory gdf_mht_pu ({len(gdf_pickups):,} rows)")
else:
    gdf_pickups = gpd.read_parquet(PICKUPS_PATH)
    print(f"Loaded pickups from {PICKUPS_PATH} ({len(gdf_pickups):,} rows)")

if gdf_pickups.crs is None:
    raise ValueError("Pickups GeoDataFrame has no CRS.")
if str(gdf_pickups.crs) != TARGET_CRS:
    gdf_pickups = gdf_pickups.to_crs(TARGET_CRS)

# 2. Load road network if not already in memory
if "G" in globals() and "edges" in globals():
    G = globals()["G"]
    edges = globals()["edges"].copy()
    print(f"Using in-memory G and edges ({len(edges):,} edges)")
    if str(edges.crs) != TARGET_CRS:
        edges = edges.to_crs(TARGET_CRS)
        nodes_gdf, edges_idx = ox.graph_to_gdfs(G)
        nodes_gdf = nodes_gdf.to_crs(TARGET_CRS)
        nodes_gdf["x"] = nodes_gdf.geometry.x
        nodes_gdf["y"] = nodes_gdf.geometry.y
        G = ox.graph_from_gdfs(
            nodes_gdf.set_index("osmid", drop=False),
            edges.to_crs(TARGET_CRS).set_index(["u", "v", "key"]),
        )
else:
    G, edges = _load_graph_from_gpkg(ROADS_PATH, NODES_PATH, TARGET_CRS)
    print(f"Loaded graph from {ROADS_PATH.name} ({len(edges):,} edges, CRS {edges.crs})")

# 3–4. Snap pickups to nearest edges in chunks; store edge_u / edge_v
n = len(gdf_pickups)
edge_u = np.empty(n, dtype=np.int64)
edge_v = np.empty(n, dtype=np.int64)

print(f"Snapping {n:,} pickups in chunks of {CHUNK_SIZE:,}...")
for start in range(0, n, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, n)
    chunk = gdf_pickups.iloc[start:end]
    X = chunk.geometry.x.to_numpy()
    Y = chunk.geometry.y.to_numpy()

    uvk = ox.nearest_edges(G, X, Y)
    edge_u[start:end] = np.fromiter((t[0] for t in uvk), dtype=np.int64, count=end - start)
    edge_v[start:end] = np.fromiter((t[1] for t in uvk), dtype=np.int64, count=end - start)

    print(f"  {end:,} / {n:,} ({100 * end / n:.1f}%)")

gdf_pickups = gdf_pickups.copy()
gdf_pickups["edge_u"] = edge_u
gdf_pickups["edge_v"] = edge_v

# 5. Pickup counts per (u, v) edge pair
demand = (
    gdf_pickups.groupby(["edge_u", "edge_v"], as_index=False)
    .size()
    .rename(columns={"size": "demand_count"})
)
print(f"Unique (u, v) pairs with demand: {len(demand):,}")

# 6–7. Join demand onto edges; fill missing with 0
edges_out = edges.merge(
    demand,
    left_on=["u", "v"],
    right_on=["edge_u", "edge_v"],
    how="left",
)
edges_out = edges_out.drop(columns=["edge_u", "edge_v"], errors="ignore")
edges_out["demand_count"] = edges_out["demand_count"].fillna(0).astype(np.int64)

# 8. Summary stats
total_edges = len(edges_out)
nonzero = int((edges_out["demand_count"] > 0).sum())
dc = edges_out["demand_count"]
print(f"Total edges: {total_edges:,}")
print(f"Edges with non-zero demand: {nonzero:,}")
print(f"demand_count min: {dc.min()}")
print(f"demand_count max: {dc.max()}")
print(f"demand_count mean: {dc.mean():.2f}")

# 9. Save enriched network
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
edges_out.to_file(OUT_PATH, driver="GPKG")
print(f"Saved -> {OUT_PATH.resolve()}")